# P05 European Climate Trends
## Notebook 01 - Data Profiling

**Project:** P05 European Climate Trends 
**Author:** Ose Omokhua

---

## Purpose

This notebook has four tasks:

1. Verify the environment and confirm all libraries import correctly
2. Register with the Copernicus Climate Data Store (CDS) API and download the E-OBS NetCDF files for mean temperature (TG) and precipitation (RR)
3. Inspect the structure of the NetCDF files - dimensions, coordinates, variables, and metadata
4. Profile data quality across all 30 countries - identify missing values by year and flag years that fail the 10% missing threshold

The E-OBS dataset is a daily gridded observational product covering Europe from 1950 to 2024 at 0.25 degree resolution. It is the primary reference dataset used by the Copernicus Climate Change Service and the European Environment Agency for monitoring climate trends across Europe.

All raw data is saved to `data/raw/` which is gitignored. Processed outputs are saved to `data/processed/`

## 1. Environment Check

We verify that all required libraries import correctly and confirm version numbers.
This is the first cell in every notebook - it catches environment problems before
any analysis begins.

In [4]:
import os
import sys
import importlib

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
import xarray as xr
import netCDF4
import cdsapi
import pymannkendall as mk

sys.path.insert(0, os.path.abspath(".."))
import config

print("Python     :", sys.version)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("SciPy      :", scipy.__version__)
print("Matplotlib :", matplotlib.__version__)
print("Seaborn    :", sns.__version__)
print("Statsmodels:", statsmodels.__version__)
print("xarray     :", xr.__version__)
print("netCDF4    :", netCDF4.__version__)
print("cdsapi     :", importlib.metadata.version("cdsapi"))
print()
print("Project root:", config.PROJECT_ROOT)
print("Data raw    :", config.DATA_RAW)
print("Figures     :", config.FIGURES_DIR)

Python     : 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
NumPy      : 2.4.4
Pandas     : 3.0.2
SciPy      : 1.17.1
Matplotlib : 3.10.8
Seaborn    : 0.13.2
Statsmodels: 0.14.6
xarray     : 2026.4.0
netCDF4    : 1.7.4
cdsapi     : 0.7.7

Project root: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5
Data raw    : C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw
Figures     : C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\figures


## 2. CDS API and E-OBS Download

The E-OBS dataset is hosted on the Copernicus Climate Data Store (CDS). We use
the `cdsapi` library to download two NetCDF files programmatically:

- **TG** - daily mean temperature (degrees Celsius), 0.25 degree resolution
- **RR** - daily precipitation sum (millimetres), 0.25 degree resolution

Both files cover the full E-OBS version 30.0e record: 1 January 1950 to
31 December 2024, over the domain 25N-71.5N by 25W-45E.

The files are large - TG is approximately 3-4 GB and RR is approximately
3-4 GB. Download time depends on CDS queue load and your connection speed.
Both files are saved to `data/raw/` which is gitignored.

The `.cdsapirc` file containing the API key must be present at
`C:\Users\TEST\.cdsapirc` before this cell runs.

In [14]:
import cdsapi
import zipfile

c = cdsapi.Client()

def download_and_extract(variable, var_label, zip_name, nc_name):
    """
    Download an E-OBS variable from CDS, save as zip, extract the NC file,
    then delete the zip.
    """
    zip_path = os.path.join(config.DATA_RAW, zip_name)
    nc_path  = os.path.join(config.DATA_RAW, nc_name)

    if os.path.exists(nc_path):
        print(f"{var_label} already exists, skipping download.")
        return

    print(f"Downloading {var_label}...")
    c.retrieve(
        "insitu-gridded-observations-europe",
        {
            "product_type":    "ensemble_mean",
            "variable":        [variable],
            "grid_resolution": "0_25deg",
            "period":          "full_period",
            "version":         ["31_0e"],
        }
    ).download(zip_path)
    print(f"{var_label} downloaded to: {zip_path}")

    print(f"Extracting {var_label}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        print("  Contents:", z.namelist())
        z.extractall(config.DATA_RAW)
    print(f"{var_label} extracted to: {config.DATA_RAW}")

    os.remove(zip_path)
    print(f"Zip deleted: {zip_path}")
    print()

# -- Temperature (TG) --------------------------------------------------------
download_and_extract(
    variable  = "mean_temperature",
    var_label = "TG",
    zip_name  = "tg_ens_mean_0.25deg_reg_v31.0e.zip",
    nc_name   = "tg_ens_mean_0.25deg_reg_v31.0e.nc"
)

# -- Precipitation (RR) ------------------------------------------------------
download_and_extract(
    variable  = "precipitation_amount",
    var_label = "RR",
    zip_name  = "rr_ens_mean_0.25deg_reg_v31.0e.zip",
    nc_name   = "rr_ens_mean_0.25deg_reg_v31.0e.nc"
)

print("All downloads and extractions complete.")

2026-04-21 02:08:42,788 INFO Request ID is bd13068b-9fbb-4ea2-914f-a2eed3efa156


2026-04-21 02:08:42,870 INFO status has been updated to accepted
2026-04-21 02:09:04,245 INFO status has been updated to successful


14a9e539c842d24cdb46b8be3cc7a67e.zip:   0%|          | 0.00/788M [00:00<?, ?B/s]

TG downloaded to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw\tg_ens_mean_0.25deg_reg_v31.0e.zip
Extracting TG...
  Contents: ['tg_ens_mean_0.25deg_reg_v31.0e.nc']
TG extracted to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw
Zip deleted: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw\tg_ens_mean_0.25deg_reg_v31.0e.zip



2026-04-21 02:14:33,481 INFO Request ID is 32fd65f8-02bf-48f4-af9b-1f6e7f2f2b61
2026-04-21 02:14:33,652 INFO status has been updated to accepted
2026-04-21 02:14:47,428 INFO status has been updated to running
2026-04-21 02:14:55,101 INFO status has been updated to successful


4b97e733a96e7acf6651b16613fc55ae.zip:   0%|          | 0.00/349M [00:00<?, ?B/s]

RR downloaded to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw\rr_ens_mean_0.25deg_reg_v31.0e.zip
Extracting RR...
  Contents: ['rr_ens_mean_0.25deg_reg_v31.0e.nc']
RR extracted to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw
Zip deleted: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\raw\rr_ens_mean_0.25deg_reg_v31.0e.zip

All downloads and extractions complete.


## 3. NetCDF Structure Inspection

Before any analysis, we inspect the structure of both NetCDF files. A NetCDF
file is a self-describing array format - it contains not just the data but
also metadata about dimensions, coordinates, and variable attributes.

For E-OBS, the structure we expect is:

- **Dimensions:** `time`, `latitude`, `longitude`
- **Coordinates:** daily time steps from 1950-01-01 to 2024-12-31, latitude
  from 25.125N to 71.375N at 0.25 degree spacing, longitude from 24.875W
  to 44.875E at 0.25 degree spacing
- **Data variable (TG):** `tg` - daily mean temperature in degrees Celsius
- **Data variable (RR):** `rr` - daily precipitation sum in millimetres

We open both files with xarray, which represents the NetCDF as a Dataset
object - a dictionary-like container of DataArrays sharing common dimensions.
We do not load the full arrays into memory at this stage. xarray uses lazy
loading - data is only read from disk when explicitly requested.

In [15]:
import zipfile

tg_path = os.path.join(config.DATA_RAW, "tg_ens_mean_0.25deg_reg_v31.0e.nc")

# Check the first 4 bytes - zip files start with PK\x03\x04
with open(tg_path, "rb") as f:
    header = f.read(4)

print("File header bytes:", header)
print("Is zip file:", zipfile.is_zipfile(tg_path))

File header bytes: b'\x89HDF'
Is zip file: False


In [17]:
# Open TG file with lazy loading - data stays on disk until needed
ds_tg = xr.open_dataset(os.path.join(config.DATA_RAW, "tg_ens_mean_0.25deg_reg_v31.0e.nc"),
                       engine="netcdf4")

print("=== TG Dataset Structure ===")
print(ds_tg)
print()
print("=== Dimensions ===")
for dim, size in ds_tg.sizes.items():
    print(f"  {dim}: {size}")
print()
print("=== Coordinates ===")
for coord in ds_tg.coords:
    print(f"  {coord}: {ds_tg.coords[coord].values[0]} to {ds_tg.coords[coord].values[-1]}")
print()
print("=== Data Variables ===")
for var in ds_tg.data_vars:
    print(f"  {var}: {ds_tg[var].attrs}")
print()
print("=== Global Attributes ===")
for attr, val in ds_tg.attrs.items():
    print(f"  {attr}: {val}")
    

=== TG Dataset Structure ===
<xarray.Dataset> Size: 10GB
Dimensions:    (time: 27394, latitude: 201, longitude: 464)
Coordinates:
  * time       (time) datetime64[ns] 219kB 1950-01-01 1950-01-02 ... 2024-12-31
  * latitude   (latitude) float64 2kB 25.38 25.62 25.88 ... 74.88 75.12 75.38
  * longitude  (longitude) float64 4kB -40.38 -40.12 -39.88 ... 75.12 75.38
Data variables:
    tg         (time, latitude, longitude) float32 10GB ...
Attributes:
    E-OBS_version:  31.0e
    Conventions:    CF-1.4
    References:     http://surfobs.climate.copernicus.eu/dataaccess/access_eo...
    history:        Mon Mar  3 16:11:16 2025: ncks --no-abc -d time,0,27393 /...
    NCO:            netCDF Operators version 5.2.2 (Homepage = http://nco.sf....

=== Dimensions ===
  time: 27394
  latitude: 201
  longitude: 464

=== Coordinates ===
  longitude: -40.375 to 75.375
  latitude: 25.375 to 75.375
  time: 1950-01-01T00:00:00.000000000 to 2024-12-31T00:00:00.000000000

=== Data Variables ===
  tg: {'u

## 4. Inspect RR (Precipitation) File

We open the precipitation file using the same approach as TG. We expect
identical dimensions - same time range, same grid - but a different data
variable (`rr`) with units of millimetres per day.

In [18]:
ds_rr = xr.open_dataset(
    os.path.join(config.DATA_RAW, "rr_ens_mean_0.25deg_reg_v31.0e.nc"),
    engine="netcdf4"
)

print("=== RR Dataset Structure ===")
print(ds_rr)
print()
print("=== Dimensions ===")
for dim, size in ds_rr.sizes.items():
    print(f"  {dim}: {size}")
print()
print("=== Coordinates ===")
for coord in ds_rr.coords:
    print(f"  {coord}: {ds_rr.coords[coord].values[0]} to {ds_rr.coords[coord].values[-1]}")
print()
print("=== Data Variables ===")
for var in ds_rr.data_vars:
    print(f"  {var}: {ds_rr[var].attrs}")
print()
print("=== Global Attributes ===")
for attr, val in ds_rr.attrs.items():
    print(f"  {attr}: {val}")

=== RR Dataset Structure ===
<xarray.Dataset> Size: 10GB
Dimensions:    (time: 27394, latitude: 201, longitude: 464)
Coordinates:
  * time       (time) datetime64[ns] 219kB 1950-01-01 1950-01-02 ... 2024-12-31
  * latitude   (latitude) float64 2kB 25.38 25.62 25.88 ... 74.88 75.12 75.38
  * longitude  (longitude) float64 4kB -40.38 -40.12 -39.88 ... 75.12 75.38
Data variables:
    rr         (time, latitude, longitude) float32 10GB ...
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF-1.4
    E-OBS_version:  31.0e
    References:     http://surfobs.climate.copernicus.eu/dataaccess/access_eo...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    history:        Tue Mar  4 09:35:28 2025: ncks --no-abc -d time,0,27393 /...
    NCO:            netCDF Operators version 5.2.2 (Homepage = http://nco.sf....

=== Dimensions ===
  time: 27394
  latitude: 201
  longitude: 464

=== Coordinates ===
  

## 5. Country Boundary Extraction

E-OBS is a gridded dataset covering the entire European domain. To compute
country-level climate statistics, we need to extract the grid points that
fall within each country's boundaries.

We use the `regionmask` library for this. `regionmask` provides standardised
country boundary masks based on Natural Earth shapefiles. For each country it
produces a boolean mask -- a 2D array of True and False values on the same
latitude-longitude grid as E-OBS -- where True indicates a grid point falls
inside that country's boundaries.

The masking approach is the correct scientific method for this task. The
alternative -- selecting grid points by bounding box -- would include ocean
points and grid points from neighbouring countries, producing biased
country-level averages.

We first check whether `regionmask` is installed, then install it if needed,
then generate masks for all 30 countries in our analysis.

In [19]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "regionmask"])

import regionmask

print("regionmask version:", regionmask.__version__)
print("Natural Earth countries available:", len(regionmask.defined_regions.natural_earth_v5_0_0.countries_110))

regionmask version: 0.13.0
Natural Earth countries available: 177


## 6. Generate Country Masks

For each of the 30 countries in `config.COUNTRIES`, we generate a boolean
mask on the E-OBS 0.25 degree grid. The mask is True where a grid point
centre falls inside the country boundary and False everywhere else.

The mask is computed once and applied repeatedly - once for TG and once for
RR - so we generate it from the latitude and longitude coordinates of the
TG dataset (which are identical to RR).

For each country we also compute the number of valid land grid points inside
its boundary. This is a data quality indicator - countries with very few
grid points (small nations like Luxembourg) will have higher spatial
uncertainty in their country-level averages than large countries like France
or Germany.

In [22]:
land110 = regionmask.defined_regions.natural_earth_v5_0_0.countries_110

lats = ds_tg.latitude.values
lons = ds_tg.longitude.values

mask = land110.mask(lons, lats)

# Build a lookup from name to region number
name_to_number = {name: num for name, num in zip(land110.names, land110.numbers)}

country_grid_counts = {}
not_found = []

for country in config.COUNTRIES:
    if country not in name_to_number:
        not_found.append(country)
        continue

    region_number = name_to_number[country]
    n_points = int((mask == region_number).sum())
    country_grid_counts[country] = n_points

print("Country grid point counts (0.25 degree resolution):")
print("-" * 45)
for country, count in sorted(country_grid_counts.items(), key=lambda x: x[1], reverse=True):
    flag = " <-- LOW" if count < 10 else ""
    print(f"  {country:<25} {count:>5} grid points{flag}")

print()
print(f"Total countries matched: {len(country_grid_counts)} of {len(config.COUNTRIES)}")
print(f"Countries not found: {not_found}")
print(f"Countries with fewer than 10 grid points: {sum(1 for v in country_grid_counts.values() if v < 10)}")

Country grid point counts (0.25 degree resolution):
---------------------------------------------
  Sweden                     1261 grid points
  Ukraine                    1123 grid points
  France                     1046 grid points
  Finland                    1024 grid points
  Norway                      992 grid points
  Spain                       852 grid points
  Germany                     744 grid points
  Poland                      648 grid points
  Italy                       551 grid points
  United Kingdom              549 grid points
  Romania                     444 grid points
  Greece                      220 grid points
  Bulgaria                    194 grid points
  Hungary                     173 grid points
  Czechia                     163 grid points
  Austria                     160 grid points
  Portugal                    159 grid points
  Latvia                      153 grid points
  Lithuania                   138 grid points
  Serbia                    

## 7. Data Quality Profiling

We now assess data quality for each of the 30 countries across the full
1950-2024 record. For each country and each year we compute:

- `n_obs` -- the number of non-missing daily observations
- `n_missing` -- the number of missing daily observations
- `pct_missing` -- the percentage of days missing
- `quality_flag` -- PASS if pct_missing is below 10%, FAIL otherwise

The 10% threshold means we tolerate up to 36 missing days per year before
flagging a country-year as unreliable. Years with more than 10% missing
data will be excluded from trend analysis for that country.

We run this profiling for TG only at this stage. The RR quality profile
will follow the same pattern and is run in the same loop for efficiency.

The `data_quality_report()` function from `src/analysis.py` handles the
per-country computation. We loop over all 30 countries, extract the
country-level time series by applying the mask, spatially average across
all valid grid points, then pass the resulting 1D series to the function.

In [25]:
tg_quality_frames = []
rr_quality_frames = []

years = list(range(config.ANALYSIS_START, config.ANALYSIS_END + 1))

# Convert mask to numpy boolean array per country outside the loop
# This avoids xarray broadcasting the mask against the data

for country in config.COUNTRIES:
    if country not in name_to_number:
        continue

    region_number  = name_to_number[country]
    country_mask_np = (mask.values == region_number)  # shape (201, 464), numpy bool

    tg_records = []
    rr_records = []

    for year in years:
        # Load one year as numpy array - shape (n_days, 201, 464)
        tg_year_np = ds_tg["tg"].sel(time=str(year)).values
        rr_year_np = ds_rr["rr"].sel(time=str(year)).values

        n_days = tg_year_np.shape[0]

        # Apply mask: set grid points outside country to NaN
        # country_mask_np is broadcast across time dimension automatically
        tg_masked = tg_year_np[:, country_mask_np]  # shape (n_days, n_country_points)
        rr_masked = rr_year_np[:, country_mask_np]

        # Spatial mean across country grid points for each day
        tg_daily_mean = np.nanmean(tg_masked, axis=1)  # shape (n_days,)
        rr_daily_mean = np.nanmean(rr_masked, axis=1)

        # Count missing days -- NaN in all grid points for that day
        tg_missing = int(np.isnan(tg_daily_mean).sum())
        rr_missing = int(np.isnan(rr_daily_mean).sum())

        tg_records.append({
            "country":      country,
            "year":         year,
            "n_obs":        n_days - tg_missing,
            "n_missing":    tg_missing,
            "pct_missing":  tg_missing / n_days * 100,
            "quality_flag": "FAIL" if tg_missing / n_days > 0.10 else "PASS"
        })
        rr_records.append({
            "country":      country,
            "year":         year,
            "n_obs":        n_days - rr_missing,
            "n_missing":    rr_missing,
            "pct_missing":  rr_missing / n_days * 100,
            "quality_flag": "FAIL" if rr_missing / n_days > 0.10 else "PASS"
        })

    tg_quality_frames.append(pd.DataFrame(tg_records))
    rr_quality_frames.append(pd.DataFrame(rr_records))
    print(f"Profiled: {country}")

tg_quality_df = pd.concat(tg_quality_frames, ignore_index=True)
rr_quality_df = pd.concat(rr_quality_frames, ignore_index=True)

print()
print("=== TG Quality Summary ===")
print(f"Total country-years assessed: {len(tg_quality_df)}")
print(f"PASS: {(tg_quality_df['quality_flag'] == 'PASS').sum()}")
print(f"FAIL: {(tg_quality_df['quality_flag'] == 'FAIL').sum()}")
print()
tg_fails = tg_quality_df[tg_quality_df["quality_flag"] == "FAIL"]
if len(tg_fails) > 0:
    print("=== TG FAIL cases ===")
    print(tg_fails[["country", "year", "pct_missing"]].to_string(index=False))
else:
    print("No TG FAIL cases.")
print()
print("=== RR Quality Summary ===")
print(f"Total country-years assessed: {len(rr_quality_df)}")
print(f"PASS: {(rr_quality_df['quality_flag'] == 'PASS').sum()}")
print(f"FAIL: {(rr_quality_df['quality_flag'] == 'FAIL').sum()}")
print()
rr_fails = rr_quality_df[rr_quality_df["quality_flag"] == "FAIL"]
if len(rr_fails) > 0:
    print("=== RR FAIL cases ===")
    print(rr_fails[["country", "year", "pct_missing"]].to_string(index=False))
else:
    print("No RR FAIL cases.")

Profiled: Austria
Profiled: Belgium
Profiled: Bulgaria
Profiled: Croatia
Profiled: Czechia
Profiled: Denmark
Profiled: Estonia
Profiled: Finland
Profiled: France
Profiled: Germany
Profiled: Greece
Profiled: Hungary
Profiled: Ireland
Profiled: Italy
Profiled: Latvia
Profiled: Lithuania
Profiled: Luxembourg
Profiled: Netherlands
Profiled: Norway
Profiled: Poland
Profiled: Portugal
Profiled: Romania
Profiled: Serbia
Profiled: Slovakia
Profiled: Slovenia
Profiled: Spain
Profiled: Sweden
Profiled: Switzerland
Profiled: Ukraine
Profiled: United Kingdom

=== TG Quality Summary ===
Total country-years assessed: 2250
PASS: 2250
FAIL: 0

No TG FAIL cases.

=== RR Quality Summary ===
Total country-years assessed: 2250
PASS: 2250
FAIL: 0

No RR FAIL cases.


## 8. Export Quality Reports

We save both quality DataFrames to `data/processed/` as CSV files. This
serves two purposes. First, the quality reports are inputs to notebook 02
-- saving them here means notebook 02 can load them directly without
re-running the profiling loop. Second, the CSVs provide a permanent audit
trail of data quality decisions -- any analyst reviewing this work can
inspect exactly which country-years were assessed and what their missing
value rates were.

In [26]:
tg_quality_path = os.path.join(config.DATA_PROCESSED, "tg_quality_report.csv")
rr_quality_path = os.path.join(config.DATA_PROCESSED, "rr_quality_report.csv")

tg_quality_df.to_csv(tg_quality_path, index=False)
rr_quality_df.to_csv(rr_quality_path, index=False)

print("TG quality report saved to:", tg_quality_path)
print("RR quality report saved to:", rr_quality_path)
print()
print("TG quality report shape:", tg_quality_df.shape)
print("RR quality report shape:", rr_quality_df.shape)
print()
print("TG quality report preview:")
print(tg_quality_df.head(10).to_string(index=False))

TG quality report saved to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\processed\tg_quality_report.csv
RR quality report saved to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\processed\rr_quality_report.csv

TG quality report shape: (2250, 6)
RR quality report shape: (2250, 6)

TG quality report preview:
country  year  n_obs  n_missing  pct_missing quality_flag
Austria  1950    365          0          0.0         PASS
Austria  1951    365          0          0.0         PASS
Austria  1952    366          0          0.0         PASS
Austria  1953    365          0          0.0         PASS
Austria  1954    365          0          0.0         PASS
Austria  1955    365          0          0.0         PASS
Austria  1956    366          0          0.0         PASS
Austria  1957    365          0          0.0         PASS
Austria  1958    365          0          0.0         PASS
Austria  1959

## 9. Annual Aggregation

We now compute two annual climate series for each country:

- **Annual mean temperature** -- the mean of all daily mean temperature
  values within a calendar year, in degrees Celsius. This is the primary
  variable for trend detection.
- **Annual total precipitation** -- the sum of all daily precipitation
  values within a calendar year, in millimetres. Summing rather than
  averaging is the correct aggregation for precipitation because we want
  to know how much rain fell in total across the year, not the average
  daily rate.

For each country we apply the same numpy boolean masking approach used in
the quality profiling loop -- selecting only grid points inside the country
boundary, then computing the spatial mean across those points to get a
single country-level value per day, then aggregating across days to get a
single value per year.

The output is two DataFrames -- one for temperature and one for
precipitation -- each with columns: country, year, and the aggregated
value. These are saved to `data/processed/` and carried forward as the
primary inputs to notebook 02.

In [32]:
tg_annual_records = []
rr_annual_records = []

years = list(range(config.ANALYSIS_START, config.ANALYSIS_END + 1))

for country in config.COUNTRIES:
    if country not in name_to_number:
        continue

    region_number   = name_to_number[country]
    country_mask_np = (mask.values == region_number)

    for year in years:
        tg_year_np = ds_tg["tg"].sel(time=str(year)).values
        rr_year_np = ds_rr["rr"].sel(time=str(year)).values

        tg_masked = tg_year_np[:, country_mask_np]
        rr_masked = rr_year_np[:, country_mask_np]

        tg_daily_mean = np.nanmean(tg_masked, axis=1)
        rr_daily_sum  = np.nansum(rr_masked,  axis=1)

        tg_annual = float(np.nanmean(tg_daily_mean))
        rr_annual = float(np.nansum(rr_daily_sum))

        tg_annual_records.append({
            "country":          country,
            "year":             year,
            "mean_temp_c":      tg_annual,
        })
        rr_annual_records.append({
            "country":          country,
            "year":             year,
            "total_precip_mm":  rr_annual,
        })

    print(f"Aggregated: {country}")

tg_annual_df = pd.DataFrame(tg_annual_records)
rr_annual_df = pd.DataFrame(rr_annual_records)

print()
print("=== TG Annual DataFrame ===")
print("Shape:", tg_annual_df.shape)
print(tg_annual_df.head(10).to_string(index=False))
print()
print("=== RR Annual DataFrame ===")
print("Shape:", rr_annual_df.shape)
print(rr_annual_df.head(10).to_string(index=False))

Aggregated: Austria
Aggregated: Belgium
Aggregated: Bulgaria
Aggregated: Croatia
Aggregated: Czechia
Aggregated: Denmark
Aggregated: Estonia
Aggregated: Finland
Aggregated: France
Aggregated: Germany
Aggregated: Greece
Aggregated: Hungary
Aggregated: Ireland
Aggregated: Italy
Aggregated: Latvia
Aggregated: Lithuania
Aggregated: Luxembourg
Aggregated: Netherlands
Aggregated: Norway
Aggregated: Poland
Aggregated: Portugal
Aggregated: Romania
Aggregated: Serbia
Aggregated: Slovakia
Aggregated: Slovenia
Aggregated: Spain
Aggregated: Sweden
Aggregated: Switzerland
Aggregated: Ukraine
Aggregated: United Kingdom

=== TG Annual DataFrame ===
Shape: (2250, 3)
country  year  mean_temp_c
Austria  1950     6.972445
Austria  1951     7.060195
Austria  1952     6.040322
Austria  1953     6.844952
Austria  1954     5.601598
Austria  1955     5.770272
Austria  1956     5.053670
Austria  1957     6.703349
Austria  1958     6.483297
Austria  1959     6.836024

=== RR Annual DataFrame ===
Shape: (2250, 3

## 10. Precipitation Normalisation

The raw `total_precip_mm` values in the RR annual DataFrame are the sum of
daily precipitation across all grid points inside each country, summed
across all days in the year. This is not directly interpretable as a
country-level precipitation value.

To convert to meaningful millimetres per year -- the standard unit for
annual precipitation in climate science -- we divide by the number of grid
points inside each country. This gives the average annual precipitation
across the country's land area, equivalent to how a single representative
station would report it.

The normalised value is:

    precip_mm_yr = total_precip_mm / n_grid_points

We add the grid point counts from the masking step, apply the
normalisation, and replace `total_precip_mm` with `precip_mm_yr` before
saving to disk.

In [33]:
# Build grid point count Series from country_grid_counts dict
grid_counts_s = pd.Series(country_grid_counts, name="n_grid_points")
grid_counts_s.index.name = "country"
grid_counts_df = grid_counts_s.reset_index()

# Merge grid point counts into RR annual DataFrame
rr_annual_df = rr_annual_df.merge(grid_counts_df, on="country", how="left")

# Normalise total precipitation by grid point count
rr_annual_df["precip_mm_yr"] = (
    rr_annual_df["total_precip_mm"] / rr_annual_df["n_grid_points"]
)

# Drop intermediate columns
rr_annual_df = rr_annual_df.drop(columns=["total_precip_mm", "n_grid_points"])

print("=== RR Annual DataFrame after normalisation ===")
print("Shape:", rr_annual_df.shape)
print(rr_annual_df.head(10).to_string(index=False))
print()
print("Sanity check -- Austria 1950 precip:")
austria_1950 = rr_annual_df[
    (rr_annual_df["country"] == "Austria") &
    (rr_annual_df["year"] == 1950)
]["precip_mm_yr"].values[0]
print(f"  Austria 1950: {austria_1950:.1f} mm/yr")
print(f"  Expected range for Austria: 700-1200 mm/yr")
print()
print("Sanity check -- United Kingdom 1980 precip:")
uk_1980 = rr_annual_df[
    (rr_annual_df["country"] == "United Kingdom") &
    (rr_annual_df["year"] == 1980)
]["precip_mm_yr"].values[0]
print(f"  United Kingdom 1980: {uk_1980:.1f} mm/yr")
print(f"  Expected range for UK: 800-1500 mm/yr")

=== RR Annual DataFrame after normalisation ===
Shape: (2250, 3)
country  year  precip_mm_yr
Austria  1950    947.275000
Austria  1951    956.094336
Austria  1952   1070.117480
Austria  1953    821.545020
Austria  1954   1154.949414
Austria  1955   1010.754297
Austria  1956   1019.655664
Austria  1957    997.328711
Austria  1958   1148.744922
Austria  1959    994.493164

Sanity check -- Austria 1950 precip:
  Austria 1950: 947.3 mm/yr
  Expected range for Austria: 700-1200 mm/yr

Sanity check -- United Kingdom 1980 precip:
  United Kingdom 1980: 921.0 mm/yr
  Expected range for UK: 800-1500 mm/yr


## 11. Export Annual Aggregations

We save both annual aggregation DataFrames to `data/processed/`. These are
the primary inputs to notebook 02 where trend detection, Mann-Kendall
testing, and Sen's slope estimation are performed.

Two files are written:

- `tg_annual.csv` -- annual mean temperature in degrees Celsius per country
  per year, 1950-2024
- `rr_annual.csv` -- normalised annual precipitation in mm per year per
  country per year, 1950-2024

In [34]:
tg_annual_path = os.path.join(config.DATA_PROCESSED, "tg_annual.csv")
rr_annual_path = os.path.join(config.DATA_PROCESSED, "rr_annual.csv")

tg_annual_df.to_csv(tg_annual_path, index=False)
rr_annual_df.to_csv(rr_annual_path, index=False)

print("TG annual data saved to:", tg_annual_path)
print("RR annual data saved to:", rr_annual_path)
print()
print("TG annual shape:", tg_annual_df.shape)
print("RR annual shape:", rr_annual_df.shape)
print()
print("TG annual -- value range check:")
print(f"  min mean_temp_c : {tg_annual_df['mean_temp_c'].min():.2f} C")
print(f"  max mean_temp_c : {tg_annual_df['mean_temp_c'].max():.2f} C")
print(f"  mean mean_temp_c: {tg_annual_df['mean_temp_c'].mean():.2f} C")
print()
print("RR annual -- value range check:")
print(f"  min precip_mm_yr : {rr_annual_df['precip_mm_yr'].min():.1f} mm")
print(f"  max precip_mm_yr : {rr_annual_df['precip_mm_yr'].max():.1f} mm")
print(f"  mean precip_mm_yr: {rr_annual_df['precip_mm_yr'].mean():.1f} mm")
print()
print("Files in data/processed/:")
for f in sorted(os.listdir(config.DATA_PROCESSED)):
    size_kb = os.path.getsize(os.path.join(config.DATA_PROCESSED, f)) / 1024
    print(f"  {f}  --  {size_kb:.1f} KB")

TG annual data saved to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\processed\tg_annual.csv
RR annual data saved to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project5\data\processed\rr_annual.csv

TG annual shape: (2250, 3)
RR annual shape: (2250, 3)

TG annual -- value range check:
  min mean_temp_c : -0.77 C
  max mean_temp_c : 16.39 C
  mean mean_temp_c: 8.63 C

RR annual -- value range check:
  min precip_mm_yr : 237.6 mm
  max precip_mm_yr : 1801.0 mm
  mean precip_mm_yr: 735.9 mm

Files in data/processed/:
  rr_annual.csv  --  70.8 KB
  rr_quality_report.csv  --  64.7 KB
  tg_annual.csv  --  71.9 KB
  tg_quality_report.csv  --  64.7 KB
